# BioHub Cell Tracking
## Notebook 05 - Inference & Submission

This notebook demonstrates how to:

- Load the trained CNN
- Run inference on unseen data
- Filter detector candidates with the CNN
- Generate predictions
- Build a Kaggle submission

Dataset -> detector -> CNN -> inference -> prediction -> submission, without retraining anything.
Every function used here is imported from `src/` (`src/detector.py`, `src/model.py`,
`src/predict.py`) — the same code `python -m src.predict` runs from the command line — so
there's nothing reimplemented in this notebook that could drift out of sync with it.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src import config
from src.dataset import BioHubDataset
from src.detector import CellDetector
from src.predict import load_model, classify_centers


## 2. Load Dataset

In [ ]:
dataset = BioHubDataset(config.DATASET_PATH)

dataset.info()


## 3. Load Trained Model

Loads `models/best_model.pth` — the checkpoint `Experiment.promote_to_default()` copies there
after every `python -m src.train` / `run_training()` call (see `src/experiment.py`), so this is
always the most recently trained model, no matter which `experiments/expNNN/` it came from.

Uses `src.predict.load_model()` rather than reimplementing checkpoint loading here, since the
checkpoint is a dict (`{"model_state_dict", "epoch", "val_loss"}`, see `src/train.py`), not a
raw state_dict — `load_model()` already knows how to unwrap that.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = load_model(config.BEST_MODEL_PATH, device)

print(f"Loaded {config.BEST_MODEL_PATH} onto {device}.")


## 4. Load Normalization Stats

The CNN was trained on patches normalized with the *training set's* mean/std (see
`src/train.py`). Using a different mean/std at inference time would silently feed
out-of-distribution values into the model, so this loads the exact stats
`Experiment.promote_to_default()` saved alongside `best_model.pth`.

In [ ]:
stats = np.load(config.BEST_NORM_STATS_PATH)
mean, std = float(stats["mean"]), float(stats["std"])

print(f"mean={mean:.4f}, std={std:.4f}")


## 5. Create Detector

In [ ]:
detector = CellDetector(
    sigma=config.GAUSSIAN_SIGMA,
    threshold_abs=config.DETECTION_THRESHOLD,
    min_distance=config.CELL_RADIUS,
)


## 6. Select Test Sample

In [ ]:
sample = dataset.test_samples[0]

volume = np.asarray(
    dataset.load_volume(sample, split="test")
)

print(sample)
print(volume.shape)


## 7. Run Detector

Loop over every frame and collect raw candidate centers. `detector.detect()` expects a 2D
max-projection (like `02_detection_baseline.ipynb` and `06_pipeline.ipynb`), not the raw 3D
frame — `CellDetector.detect_volume()` already does exactly this loop, so we reuse it instead
of writing it again.

In [ ]:
detections = detector.detect_volume(volume)

print(f"{len(detections)} frames, {sum(len(d) for d in detections)} raw detections total")


## 8. Visualize Raw Detections

Exactly like `02_detection_baseline.ipynb`: max-projection of frame 0 with raw candidate centers overlaid.

In [ ]:
mip0 = volume[0].max(axis=0)
coords0 = detections[0]

plt.figure(figsize=(8, 8))
plt.imshow(mip0, cmap="gray")
plt.scatter(coords0[:, 1], coords0[:, 0], s=30, c="red")
plt.title(f"Raw detections (frame 0): {len(coords0)} candidates")
plt.axis("off")
plt.show()


## 9. CNN Filtering

This is the important cell. The detector alone is noisy — it flags anything locally bright
enough, including debris and imaging artifacts. The CNN trained in `04_training_demo.ipynb`
looks at a small patch around each candidate center and decides whether it's really a cell.

```
Detector -> Extract patch -> CNN -> Probability -> Keep / Reject
```

`classify_centers()` (from `src/predict.py`) does exactly this for a list of centers — it's
the same function `python -m src.predict` uses, so filtering here matches the CLI exactly.

In [ ]:
kept0, confidences0 = classify_centers(model, mip0, coords0, mean, std, device)

print(f"Raw detections:      {len(coords0)}")
print(f"Filtered detections: {len(kept0)}")


## 10. Compare Before / After

This is one of the nicest figures in the project.

In [ ]:
kept0_arr = np.array(kept0) if kept0 else np.empty((0, 2))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(mip0, cmap="gray")
axes[0].scatter(coords0[:, 1], coords0[:, 0], s=30, c="red")
axes[0].set_title(f"Raw detector ({len(coords0)})")
axes[0].axis("off")

axes[1].imshow(mip0, cmap="gray")
if len(kept0_arr):
    axes[1].scatter(kept0_arr[:, 1], kept0_arr[:, 0], s=30, c="lime")
axes[1].set_title(f"After CNN ({len(kept0_arr)})")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## 11. Build Prediction DataFrame

Run detector + CNN filtering across every frame in the volume to build the full submission table.

In [ ]:
import pandas as pd

predictions = []
n_raw = 0

for t, coords_t in enumerate(detections):

    n_raw += len(coords_t)

    projection_t = volume[t].max(axis=0)
    kept_t, _ = classify_centers(model, projection_t, coords_t, mean, std, device)

    for y, x in kept_t:
        predictions.append((t, y, x))

submission = pd.DataFrame(predictions, columns=["frame", "y", "x"])

submission.head()


## 12. Save Submission

In [ ]:
config.OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

submission_path = config.OUTPUT_PATH / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Saved to {submission_path}")


## 13. Final Summary

In [ ]:
print("=================================")
print()
print("Sample:")
print(sample)
print()
print("Frames:")
print(volume.shape[0])
print()
print("Raw detections:")
print(n_raw)
print()
print("Filtered detections:")
print(len(submission))
print()
print("Submission saved to")
print(submission_path)
print()
print("=================================")
